# Who's the Best Pitcher? - retrieval + Gemma re-ranking

A hybrid. Deterministic code finds candidate facts and emits the final string;
a language model only chooses *which* candidate answers the question.

### Why not let the model write the answer

Scoring is exact string match and the rules are explicit that `123.4500` must
not become `123.45`. A generative model reformats numbers constantly, and it
cannot see 1.7 GB of JSON in context anyway. So the value is always copied
verbatim from the source file and never passes through the model.

### Why not pure keyword matching

That is what the previous version did, and it reached 0.543. Its weakness is
comprehension: the statistics live in nested groups with two-letter leaf names
(`onbase.s`, `runs.earned`, `outs.gofo`), and phrases like "on-base percentage
from singles" need to be understood rather than pattern-matched.

### The split

1. resolve the entity (team or player) and flatten its JSON subtree
2. score leaves heuristically and keep the best few as candidates
3. **Gemma picks the index of the candidate that answers the question**
4. emit that candidate's value verbatim

If the model's reply cannot be parsed, the heuristic top-1 is used, so this can
only match or improve on the deterministic pipeline.

**Settings:** GPU ON. Add the `gemma-3` model as an input. Internet may stay off.


In [ ]:
import csv, json, re
from pathlib import Path

import torch

IN = Path("/kaggle/input")


def find(pattern: str) -> Path:
    hits = sorted(IN.rglob(pattern))
    assert hits, f"could not find {pattern}"
    return max(hits, key=lambda p: p.stat().st_size)


TEAMS, LEAGUE = find("teams.json"), find("league.json")
PLAYERS, TEST = find("players.json"), find("test.csv")
TOPK_CANDIDATES = 8
print("teams  ", TEAMS)
print("players", PLAYERS, f"({PLAYERS.stat().st_size/2**30:.2f} GB)")

## Knowledge base

`parse_float=str` everywhere: `json.load` would turn `123.4500` into a float and
the trailing zeros would be gone before scoring ever sees them.


In [ ]:
STR_PARSE = {"parse_float": str, "parse_int": str, "parse_constant": str}


def load_exact(path: str | Path) -> dict:
    """Load JSON preserving the literal text of every number."""
    with open(path) as f:
        return json.load(f, **STR_PARSE)


def _iter_top_level(path: str | Path):
    """Yield (key, raw_json_text) for each entry of a top-level JSON object.

    players.json is 1.7 GB. Parsing it whole costs many gigabytes of dicts, and
    the streaming libraries are not in Kaggle's image, so this walks the file in
    chunks tracking brace depth and hands back each value as raw text. Callers
    then json.loads only the entries they need -- with parse_float=str, so the
    literal digits survive.
    """
    with open(path, "r") as f:
        buf, depth, in_str, esc = "", 0, False, False
        key, val_start, seeking_key = None, None, True
        pos = 0
        while True:
            chunk = f.read(1 << 20)
            if not chunk:
                break
            buf += chunk
            i = pos
            while i < len(buf):
                ch = buf[i]
                if in_str:
                    if esc:
                        esc = False
                    elif ch == "\\":
                        esc = True
                    elif ch == '"':
                        in_str = False
                        if depth == 1 and seeking_key:
                            key = json.loads(buf[key_start : i + 1])
                            seeking_key = False
                elif ch == '"':
                    in_str = True
                    if depth == 1 and seeking_key:
                        key_start = i
                elif ch == "{":
                    depth += 1
                    if depth == 2 and key is not None and val_start is None:
                        val_start = i
                elif ch == "}":
                    depth -= 1
                    if depth == 1 and val_start is not None:
                        yield key, buf[val_start : i + 1]
                        buf = buf[i + 1 :]
                        i, key, val_start, seeking_key = -1, None, None, True
                    elif depth == 0:
                        return
                elif ch == "," and depth == 1:
                    seeking_key = True
                i += 1
            pos = max(len(buf) - 1, 0) if val_start is None else len(buf)
            if val_start is not None:
                pos = len(buf)


def stream_player_names(path: str | Path) -> list[str]:
    """Root keys of players.json without materialising the file."""
    return [key for key, _ in _iter_top_level(path)]


def stream_players(path: str | Path, wanted: set[str]) -> dict:
    """Load only the named players, preserving every number's literal text."""
    out: dict = {}
    for key, raw in _iter_top_level(path):
        if key in wanted:
            out[key] = json.loads(raw, **STR_PARSE)
            if len(out) == len(wanted):
                break
    return out


def flatten(obj, prefix: tuple = ()) -> list[tuple[tuple, str]]:
    """Depth-first (path, leaf-value) pairs.

    Leaves are returned as strings because that is what the submission needs;
    nothing is reformatted on the way out.
    """
    out: list[tuple[tuple, str]] = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "_comment":
                continue
            out.extend(flatten(v, prefix + (str(k),)))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            out.extend(flatten(v, prefix + (str(i),)))
    elif obj is not None:
        out.append((prefix, obj if isinstance(obj, str) else str(obj)))
    return out


def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def team_aliases(teams: dict) -> dict[str, str]:
    """Map every way a team is written in a question to its root key.

    Questions use the full "Cincinnati Reds", but also bare "Reds", the market
    "Cincinnati", and abbreviations, so all of them index the same entry.
    """
    alias: dict[str, str] = {}
    for key, blob in teams.items():
        profile = blob.get("data", {}).get("team_profile", {})
        market, name, abbr = (profile.get("market", ""), profile.get("name", ""),
                              profile.get("abbr", ""))
        for form in (f"{market} {name}", name, abbr, key.replace("_", " ")):
            if form and form.strip():
                alias.setdefault(form.lower().strip(), key)
    return alias

## Heuristic candidate scoring

This no longer has to be right -- only good enough to put the correct leaf in
the top few, which the model then picks from.


In [ ]:
# question wording -> path tokens that should be treated as a match
SYNONYMS: dict[str, tuple[str, ...]] = {
    "abbreviation": ("abbr",),
    "innings": ("ip", "ip_1", "ip_2"),
    "walks": ("bb",),
    "intentional": ("ibb",),
    "strikeouts": ("so", "ktotal", "k"),
    "saves": ("sv", "svo", "save"),
    "runs": ("runs", "r"),
    "hits": ("h", "hits"),
    "homers": ("hr",),
    "home run": ("hr",),
    "home runs": ("hr",),
    "home": ("home",),
    "average": ("avg", "oba", "obp"),
    "batting": ("avg", "hitting"),
    "slugging": ("slg",),
    "on-base": ("obp", "oba"),
    "percentage": ("pct", "obp", "slg", "fpct"),
    "era": ("era",),
    "whip": ("whip",),
    "ratio": ("gofo", "kbb", "bbk"),
    "at-bats": ("ab",),
    "batters": ("bf",),
    "faced": ("bf",),
    "pitch": ("pitch_count", "pitching"),
    "count": ("pitch_count", "count"),
    "starters": ("starters",),
    "bullpen": ("bullpen",),
    "overall": ("overall",),
    "salary": ("salary",),
    "position": ("position", "primary_position"),
    "owner": ("owner",),
    "founded": ("founded",),
    "market": ("market",),
    "mascot": ("mascot",),
    "sponsor": ("sponsor",),
    "song": ("fight_song",),
    "affiliates": ("minorleague_affiliate",),
    "affiliate": ("minorleague_affiliate",),
    "reference": ("reference", "id"),
    "injured": ("injuries",),
    "elimination": ("elimination_number", "elim"),
    "wild": ("wild_card",),
    "back": ("games_back", "gb"),
    "wins": ("win", "wins", "w"),
    "losses": ("loss", "losses", "l"),
    "games": ("games", "g", "gp"),
    "played": ("gp", "games_played"),
    "started": ("gs", "games_start"),
    "preseason": ("pre",),
    "spring": ("pre",),
    "regular": ("reg",),
    "season": ("season", "reg"),
    "singles": ("single", "singles"),
    "grounded": ("go",),
    "fly": ("fo",),
    # handedness splits are keyed by a single letter in the data
    "left-handed": ("l",),
    "right-handed": ("r",),
    "lefties": ("l",),
    "righties": ("r",),
}

MONTHS = {m.lower(): i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], start=1)}

ORDINALS = {"previous": 1, "two": 2, "three": 3, "four": 4, "five": 5,
            "2": 2, "3": 3, "4": 4, "5": 5, "last": 1}

STOP = {"the", "a", "an", "of", "in", "for", "is", "was", "what", "how", "many",
        "did", "do", "does", "are", "were", "has", "have", "had", "with",
        "and", "to", "by", "at", "on", "s", "their", "his", "her", "number"}


def tokens(text: str) -> list[str]:
    return [t for t in re.split(r"[^a-z0-9_\-]+", text.lower()) if t and t not in STOP]


def expand(question_tokens: list[str]) -> set[str]:
    """Question tokens plus the path tokens they imply."""
    out = set(question_tokens)
    for t in question_tokens:
        out.update(SYNONYMS.get(t, ()))
    return out


def score_path(path: tuple[str, ...], q_expanded: set[str], q_raw: list[str]) -> float:
    """How well one leaf path matches the question.

    Deeper agreement is worth more than shallow: matching the leaf name itself
    ("era") should outweigh matching a container ("statistics").
    """
    score = 0.0
    for depth, part in enumerate(path):
        weight = 1.0 + depth / max(len(path) - 1, 1)      # later segments matter more
        parts = re.split(r"[_\s]+", part.lower())
        for p in parts:
            if not p:
                continue
            if p in q_expanded:
                score += 2.0 * weight
            elif len(p) > 3 and any(p in q or q in p for q in q_raw):
                score += 0.8 * weight                      # partial: "abbr" in "abbreviation"
    # a year mentioned in the question must appear in the path if the path has one
    years_q = {t for t in q_raw if re.fullmatch(r"(19|20)\d{2}", t)}
    years_p = {p for p in path if re.fullmatch(r"(19|20)\d{2}", p)}
    if years_q and years_p:
        score += 6.0 if (years_q & years_p) else -6.0
    # likewise months
    months_q = {t for t in q_raw if t in MONTHS}
    if months_q:
        joined = " ".join(path).lower()
        if any(m in joined for m in months_q):
            score += 5.0
    return score


# multi-word phrases that map to a single stat key; checked before single words
# Multi-word phrases that name one specific leaf. Checked before single words so
# that "intentional walks" resolves to ibb rather than bb.
#
# Most of these exist because the statistics are not flat: they sit in nested
# groups (onbase, runs, outs, outcome, steal, in_play, games, pitches) whose
# leaf names are two-letter codes. Without this mapping a question about
# "singles" or "earned runs" scores no better than the generic obp or run total
# sitting a level above it.
PHRASES: dict[str, str] = {
    # onbase group
    "intentional walk": "ibb",
    "from singles": "s",
    "singles": "s",
    "doubles": "d",
    "triples": "t",
    "total bases": "tb",
    "hit by pitch": "hbp",
    "fielder's choice": "fc",
    "reached on error": "roe",
    # runs group
    "earned run total": "earned",
    "earned runs": "earned",
    "unearned run": "unearned",
    "inherited runner": "ir",
    # strikeout flavours
    "strikeouts looking": "klook",
    "strikeout looking": "klook",
    "strikeouts swinging": "kswing",
    "strikeout swinging": "kswing",
    "total strikeout": "ktotal",
    "strikeout count": "ktotal",
    "strikeouts": "ktotal",
    # outs / batted-ball
    "ground out to fly out": "gofo",
    "ground out": "go",
    "grounded out": "go",
    "fly out": "fo",
    "flied out": "fo",
    "double play": "gidp",
    "sacrifice fly": "sacfly",
    "sacrifice hit": "sachit",
    "line drive": "linedrive",
    "ground ball": "groundball",
    "fly ball": "flyball",
    "pop up": "popup",
    "batted balls in play": "bip",
    # steal group
    "caught stealing": "caught",
    "stolen base": "stolen",
    "pickoff": "pickoff",
    # games group
    "complete game": "complete",
    "quality start": "qstart",
    "shutout": "shutout",
    "blown save": "blown_save",
    "games started": "start",
    "games played": "play",
    "did not start": "start",
    "save opportunit": "svo",
    "svo": "svo",
    "holds": "hold",
    # pitches
    "pitch count": "count",
    "pitches per inning": "per_ip",
    "play count": "play",
    # rate stats
    "earned run average": "era",
    "on-base percentage": "obp",
    "on base percentage": "obp",
    "on-base average": "oba",
    "slugging percentage": "slg",
    "batting average": "avg",
    "at-bat": "ab",
    "batters faced": "bf",
    "runs batted in": "rbi",
    "extra base hit": "xbh",
    "isolated power": "iso",
    "secondary average": "seca",
    "left on base": "lob",
    "range factor": "fpct",
    "innings pitched": "ip_1",
    # team profile
    "games back": "games_back",
    "wild card": "wild_card",
    "fight song": "fight_song",
    "minor league affiliate": "minorleague_affiliate",
    "home win": "win",
    "home loss": "loss",
    "home run": "hr",
}


def phrase_keys(question: str) -> set[str]:
    """Leaf names implied by phrases in the question, longest match wins.

    Overlapping phrases must not both fire: "ground out to fly out ratio"
    contains "fly out", and letting the shorter one through makes `fo` compete
    with the `gofo` the question actually asks for.
    """
    q = question.lower()
    hits: list[tuple[int, int, str]] = []
    for phrase, key in PHRASES.items():
        pos = q.find(phrase)
        if pos >= 0:
            hits.append((pos, len(phrase), key))
    hits.sort(key=lambda h: -h[1])          # longest first
    claimed: list[tuple[int, int]] = []
    keys: set[str] = set()
    for pos, length, key in hits:
        span = (pos, pos + length)
        if any(pos < c_end and span[1] > c_start for c_start, c_end in claimed):
            continue                        # inside a longer phrase already taken
        claimed.append(span)
        keys.add(key)
    return keys


RECENCY = {
    "previous match": "previous match",
    "last match": "previous match",
    "last appearance": "previous match",
    "two matches ago": "2 matches ago",
    "2 matches ago": "2 matches ago",
    "three matches ago": "3 matches ago",
    "3 matches ago": "3 matches ago",
    "four matches ago": "4 matches ago",
    "4 matches ago": "4 matches ago",
    "five matches ago": "5 matches ago",
    "5 matches ago": "5 matches ago",
}


def recency_key(question: str) -> str | None:
    """"the game five matches ago" names a specific last_10_games entry."""
    q = question.lower()
    for phrase, key in RECENCY.items():
        if phrase in q:
            return key
    return None


def wants_number(question: str) -> bool:
    q = question.lower()
    return (q.startswith("how many") or "number of" in q
            or any(w in q for w in ("ratio", "percentage", "average", "count",
                                    "salary", "total")))


def is_number(value: str) -> bool:
    return bool(re.fullmatch(r"-?\d+(\.\d+)?", value.strip()))


# asking about these means the answer lives in team_profile, not in any split
PROFILE_WORDS = {"owner", "founded", "market", "mascot", "sponsor", "nickname",
                 "abbreviation", "song", "president", "manager", "championship",
                 "affiliate", "venue", "stadium", "league", "division"}

UUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-", re.I)


HAND_RE = re.compile(r"\b(left|right)-handed\b", re.I)
DATE_RE = re.compile(
    r"\b(january|february|march|april|may|june|july|august|september|october|"
    r"november|december)\s+(\d{1,2}),?\s*(\d{4})", re.I)


def handedness(question: str) -> str | None:
    m = HAND_RE.search(question)
    return None if not m else ("l" if m.group(1).lower() == "left" else "r")


def explicit_date(question: str) -> str | None:
    """"on August 23, 2025" -> "2025-08-23", to pick a game by date."""
    m = DATE_RE.search(question)
    if not m:
        return None
    month = MONTHS[m.group(1).lower()]
    return f"{m.group(3)}-{month:02d}-{int(m.group(2)):02d}"


def best_leaf(subtree_flat, question: str, opponent: str | None = None,
              want_player: bool = False, entity_tokens: set[str] | None = None,
              player_slug: str | None = None, game_anchor: str | None = None):
    """Highest-scoring (path, value) for this question, or None.

    `entity_tokens` are dropped from the question before scoring. The entity has
    already been resolved, so letting "Suarez" match the path segment
    `roster.ranger_suarez` just rewards every leaf that happens to sit under his
    name -- including his preferred_name -- over the one holding the statistic
    actually asked for.
    """
    q_raw = tokens(question)
    if entity_tokens:
        q_raw = [t for t in q_raw if t not in entity_tokens]
    q_exp = expand(q_raw) | phrase_keys(question)
    forced = phrase_keys(question)
    recency = recency_key(question)
    numeric = wants_number(question)
    asks_id = any(w in question.lower() for w in ("reference number", " id ", "identifier"))
    profile_q = bool(set(q_raw) & PROFILE_WORDS)
    asks_name = question.lower().lstrip().startswith("who") or "name of" in question.lower()
    hand = handedness(question)
    on_date = explicit_date(question)
    asks_salary = "salary" in question.lower()
    asks_jersey = "jersey" in question.lower()
    best, best_score = None, 0.0
    for path, value in subtree_flat:
        s = score_path(path, q_exp, q_raw)
        leaf = path[-1].lower() if path else ""
        if forced:
            # a phrase names one specific stat -- reward that leaf, penalise rivals
            s += 8.0 if leaf in forced else -3.0
        if opponent:
            in_path = opponent in "/".join(path).lower()
            if recency:
                # "in the previous match against X" -- the game already pins the
                # opponent, so an opponent split is not expected in the path
                s += 6.0 if in_path else 0.0
            else:
                # "against X in the 2023 season" must come from that opponent's
                # split; a season total or a handedness split answers a
                # different question entirely
                s += 12.0 if in_path else -12.0
        # A team-level question must not be answered from one player's box score
        # nested inside a game summary, and vice versa.
        inside_player = "players" in [p.lower() for p in path]
        if inside_player and not want_player:
            s -= 8.0
        if player_slug:
            # The named player's own statistics outrank the surrounding team's.
            # Name tokens are excluded from generic scoring (see docstring), so
            # this bonus has to be applied explicitly rather than falling out of
            # token overlap.
            s += 9.0 if player_slug in [p.lower() for p in path] else 0.0
        if recency:
            # anchor to the right game; without this a "previous match" question
            # can drift into season-long or venue metadata
            s += 10.0 if recency in [p.lower() for p in path] else -4.0
        if numeric:
            # "how many" cannot be answered by a stadium surface or a team name
            s += 2.0 if is_number(value) else -6.0
        # Identifiers are never the answer unless explicitly requested, and a
        # leaf whose value merely repeats a path segment is a label, not data.
        if UUID_RE.match(value) and not asks_id:
            s -= 20.0
        if leaf in ("id", "sr_id", "reference") and not asks_id:
            s -= 12.0
        if value.lower() in {p.lower() for p in path}:
            s -= 15.0
        # Roster metadata (preferred_name, jersey, position...) sits under the
        # same player node as the statistics, so it has to be ruled out
        # explicitly when the question asks for a number or a named stat.
        if leaf.endswith("_name") or leaf in ("first_name", "last_name", "full_name",
                                              "preferred_name", "jersey_number"):
            if not asks_name:
                s -= 14.0
        if profile_q:
            s += 8.0 if "team_profile" in [p.lower() for p in path] else -2.0
        lower_path = [p.lower() for p in path]
        if hand:
            # the split is keyed 'l'/'r' under hitter_hand or pitcher_hand;
            # picking the wrong letter silently answers the opposite question
            if any(h in lower_path for h in ("hitter_hand", "pitcher_hand")):
                s += 8.0 if hand in lower_path else -10.0
        if game_anchor:
            # the question named a calendar date; only that game's subtree counts
            s += 12.0 if game_anchor in path else -8.0
        if asks_salary:
            s += 14.0 if leaf == "salary" else -4.0
        if asks_jersey:
            s += 14.0 if leaf in ("jersey_number", "jersey") else -4.0
        if asks_id:
            # the player's own profile id, not a per-game roster entry
            s += 15.0 if "player_profile" in lower_path else -8.0
        if s > best_score:
            best, best_score = (path, value), s
    return best, best_score


def top_leaves(subtree_flat, question: str, k: int = 5, **kw):
    """The k best-scoring leaves, for inspecting ranking quality."""
    scored = []
    for path, value in subtree_flat:
        best, sc = best_leaf([(path, value)], question, **kw)
        if best is not None:
            scored.append((sc, path, value))
    scored.sort(key=lambda t: -t[0])
    return scored[:k]

## Entity resolution


In [ ]:
def game_key_for_date(flat, iso_date: str) -> str | None:
    """Which last_10_games entry was played on `iso_date`.

    The date is stored as the value of a `scheduled` leaf, not as a path
    segment, so it cannot be matched by path scoring alone -- the containing
    game key has to be resolved first and then used as an anchor.
    """
    for path, value in flat:
        if path and path[-1] == "scheduled" and isinstance(value, str) \
                and value.startswith(iso_date):
            for seg in path:
                if "matches ago" in seg or seg == "previous match":
                    return seg
    return None

NO_ANSWER = "no answer"


def locate_teams(question: str, aliases: dict[str, str]) -> list[tuple[int, str]]:
    """Every team mentioned, as (character position, root key), left to right.

    Longest alias wins at each position so that "Chicago White Sox" is not
    captured by the shorter "Chicago" belonging to the Cubs.
    """
    q = question.lower()
    found: dict[int, tuple[int, str]] = {}
    for alias, key in aliases.items():
        for m in re.finditer(rf"\b{re.escape(alias)}\b", q):
            start = m.start()
            prev = found.get(start)
            if prev is None or len(alias) > prev[0]:
                found[start] = (len(alias), key)
    # drop mentions nested inside a longer one ("Chicago" inside "Chicago Cubs")
    spans = sorted((pos, ln, key) for pos, (ln, key) in found.items())
    kept: list[tuple[int, str]] = []
    for pos, ln, key in spans:
        if kept and pos < kept[-1][0] + len(kept[-1][1]):
            continue
        kept.append((pos, key))
    return kept


def find_team(question: str, aliases: dict[str, str],
              subject_is_player: bool = False) -> tuple[str | None, str | None]:
    """Return (subject team, opponent team).

    "How many X did the Mets have against the White Sox" names two teams: the
    subject whose statistics are being asked about, and the opponent that
    selects a split. Taking the longest alias would pick the White Sox and
    silently answer a different question, so the word "against" decides roles.
    """
    mentions = locate_teams(question, aliases)
    if not mentions:
        return None, None
    q = question.lower()
    against = [m.end() for m in re.finditer(r"\bagainst\b", q)]
    if len(mentions) == 1:
        # "<player>'s OPS against <team>" names one team, and it is the
        # opponent, not the subject -- the subject is the player. Treating it as
        # the subject silently drops the opponent split and answers with a
        # season total instead.
        pos, key = mentions[0]
        if subject_is_player and against and pos > against[0]:
            return None, key
        return key, None
    if against:
        after = [(pos, key) for pos, key in mentions if pos > against[0]]
        before = [(pos, key) for pos, key in mentions if pos < against[0]]
        if after and before:
            return before[-1][1], after[0][1]
        if after:
            return None, after[0][1]
    return mentions[0][1], mentions[-1][1]


def find_player(question: str, names: list[str]) -> str | None:
    """Root keys are slugs ("miles_mikolas"); questions spell names out."""
    q = question.lower()
    hit, hit_len = None, 0
    for name in names:
        spelled = name.replace("_", " ").lower()
        if len(spelled) > hit_len and re.search(rf"\b{re.escape(spelled)}\b", q):
            hit, hit_len = name, len(spelled)
    return hit


def run() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--teams", default="teams.json")
    ap.add_argument("--league", default="league.json")
    ap.add_argument("--players", default="players.json")
    ap.add_argument("--test", default="test.csv")
    ap.add_argument("--out", default="submission_pred.csv")
    ap.add_argument("--min-score", type=float, default=0.0,
                    help="a wrong guess scores the same as 'no answer', so guess by default")
    ap.add_argument("--audit", default="audit.csv",
                    help="write question -> chosen path -> answer, for error analysis")
    ap.add_argument("--debug", type=int, default=0, help="print the first N resolutions")
    args = ap.parse_args()

    questions = list(csv.DictReader(open(args.test)))
    print(f"{len(questions)} questions")

    teams = load_exact(args.teams)
    league = load_exact(args.league)
    aliases = team_aliases(teams)
    print(f"teams {len(teams)}  aliases {len(aliases)}")

    players: dict = {}
    if Path(args.players).exists():
        from kb import stream_player_names
        all_names = stream_player_names(args.players)
        wanted = set()
        for n in all_names:
            spelled = n.replace("_", " ").lower()
            if any(spelled in q["question"].lower() for q in questions):
                wanted.add(n)
        print(f"players in file {len(all_names)}, referenced by questions {len(wanted)}")
        players = stream_players(args.players, wanted)
    else:
        print("players.json missing -- team/league questions only")

    league_flat = flatten(league)
    team_flat: dict[str, list] = {}
    player_flat: dict[str, list] = {}

    answers, audit = [], []
    stats = {"team": 0, "player": 0, "league": 0, "none": 0}
    for row in questions:
        q = row["question"]
        player = find_player(q, list(players)) if players else None
        team, opponent = find_team(q, aliases, subject_is_player=player is not None)

        if player:
            flat = player_flat.setdefault(player, flatten(players[player]))
            source = "player"
        elif team:
            flat = team_flat.setdefault(team, flatten(teams[team]))
            source = "team"
        else:
            flat, source = league_flat, "league"

        ent_tokens: set[str] = set()
        if player:
            ent_tokens |= set(player.lower().split("_"))
        if team:
            ent_tokens |= set(team.lower().split("_"))
        if opponent:
            ent_tokens |= set(opponent.lower().split("_"))
        anchor = None
        iso = explicit_date(q)
        if iso:
            anchor = game_key_for_date(flat, iso)
        best, score = best_leaf(flat, q, opponent=opponent, game_anchor=anchor,
                                 want_player=player is not None,
                                 entity_tokens=ent_tokens,
                                 player_slug=player)
        if best is None or score < args.min_score:
            answers.append(NO_ANSWER)
            stats["none"] += 1
            audit.append((source, "-", NO_ANSWER, f"{score:.1f}"))
        else:
            answers.append(best[1])
            stats[source] += 1
            audit.append((source, ".".join(best[0]), best[1], f"{score:.1f}"))

        if args.debug and len(answers) <= args.debug:
            path = ".".join(best[0]) if best else "-"
            print(f"  [{row['ID']}] {q[:70]}")
            print(f"        -> {source}:{path}  = {answers[-1]!r}  (score {score:.1f})")

    if args.audit:
        with open(args.audit, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["ID", "question", "source", "path", "answer", "score"])
            for row, rec in zip(questions, audit):
                w.writerow([row["ID"], row["question"], *rec])
        print(f"wrote audit trail to {args.audit}")

    with open(args.out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["ID", "ANSWER"])
        for row, ans in zip(questions, answers):
            w.writerow([row["ID"], ans])

    print(f"\nresolved via {stats}")
    print(f"wrote {args.out}")

## Load the data and build candidates


In [ ]:
questions = list(csv.DictReader(open(TEST)))
teams, league = load_exact(TEAMS), load_exact(LEAGUE)
aliases = team_aliases(teams)

all_names = stream_player_names(PLAYERS)
wanted = {n for n in all_names
          if n.replace("_", " ").lower() in " || ".join(q["question"].lower() for q in questions)}
players = stream_players(PLAYERS, wanted)
print(f"{len(questions)} questions, {len(wanted)} players referenced")

league_flat = flatten(league)
team_flat, player_flat = {}, {}


def candidates(q: str, k: int = TOPK_CANDIDATES):
    """Top-k (path, value) leaves plus the context needed to score them."""
    player = find_player(q, list(players)) if players else None
    team, opponent = find_team(q, aliases, subject_is_player=player is not None)
    if player:
        flat = player_flat.setdefault(player, flatten(players[player]))
    elif team:
        flat = team_flat.setdefault(team, flatten(teams[team]))
    else:
        flat = league_flat

    ent = set()
    for e in (player, team, opponent):
        if e:
            ent |= set(e.lower().split("_"))
    iso = explicit_date(q)
    anchor = game_key_for_date(flat, iso) if iso else None

    scored = []
    for path, value in flat:
        best, sc = best_leaf([(path, value)], q, opponent=opponent,
                             want_player=player is not None, entity_tokens=ent,
                             player_slug=player, game_anchor=anchor)
        if best is not None:
            scored.append((sc, path, value))
    scored.sort(key=lambda t: -t[0])
    # de-duplicate identical values: several paths often hold the same number,
    # and showing the model eight copies wastes the candidate budget
    seen, out = set(), []
    for sc, path, value in scored:
        if value in seen:
            continue
        seen.add(value)
        out.append((path, value))
        if len(out) >= k:
            break
    return out


cand_map = {row["ID"]: candidates(row["question"]) for row in questions}
print("candidates built;", sum(len(v) for v in cand_map.values()), "total")

## Gemma picks the candidate

Greedy decoding and a pinned model version keep this reproducible. The model
emits only an index -- never the answer text.


In [ ]:
from transformers import AutoProcessor

def find_model_dir() -> str:
    """Directory that actually holds the weights.

    Kaggle nests model inputs several levels deep and the exact layout varies
    (/kaggle/input/models/google/gemma-3/transformers/... in some sessions), so
    match on the presence of a config plus a weights file rather than guessing
    the path.
    """
    for cfg in sorted(IN.rglob("config.json")):
        d = cfg.parent
        if any(d.glob("*.safetensors")) or any(d.glob("pytorch_model*.bin")):
            return str(d)
    raise FileNotFoundError(
        "No model weights under /kaggle/input -- add a Gemma model as an input "
        "(Add Input -> Models -> gemma 3 -> Transformers -> gemma-3-4b-it)")


MODEL_ID = find_model_dir()
print("model:", MODEL_ID)
print("  files:", sorted(p.name for p in Path(MODEL_ID).iterdir())[:8])

# Gemma 3 needs its own class; anything else loads through the generic one, so
# the notebook still runs if only Gemma 1/2 is available in this region.
cfg = json.loads((Path(MODEL_ID) / "config.json").read_text())
is_gemma3 = "gemma3" in cfg.get("model_type", "").lower()
print("model_type:", cfg.get("model_type"))

if is_gemma3:
    from transformers import Gemma3ForConditionalGeneration
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16).eval()
else:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16).eval()

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    tokenizer = getattr(processor, "tokenizer", processor)
except Exception:
    from transformers import AutoTokenizer
    processor = tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def choose(question: str, cands) -> int:
    listing = "\n".join(
        f"{i}. {'.'.join(p)} = {v}" for i, (p, v) in enumerate(cands))
    prompt = (
        "You are matching a baseball question to the correct field from a JSON "
        "database. Each candidate shows its full path and value.\n\n"
        f"Question: {question}\n\nCandidates:\n{listing}\n\n"
        "Which candidate answers the question? Reply with the number only."
    )
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        messages if is_gemma3 else [{"role": "user", "content": prompt}],
        add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:],
                             skip_special_tokens=True)
    m = re.search(r"\d+", reply)
    idx = int(m.group()) if m else 0
    return idx if 0 <= idx < len(cands) else 0     # fall back to the heuristic top-1


answers, changed = {}, 0
for n, row in enumerate(questions, 1):
    cands = cand_map[row["ID"]]
    if not cands:
        answers[row["ID"]] = "no answer"
        continue
    idx = choose(row["question"], cands)
    changed += idx != 0
    answers[row["ID"]] = cands[idx][1]
    if n % 40 == 0:
        print(f"  {n}/{len(questions)}  model overrode the heuristic {changed} times", flush=True)

print(f"\nmodel disagreed with the heuristic on {changed}/{len(questions)}")

## Write the submission


In [ ]:
with open("/kaggle/working/submission.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["ID", "ANSWER"])
    for row in questions:
        w.writerow([row["ID"], answers[row["ID"]]])

rows = list(csv.DictReader(open("/kaggle/working/submission.csv")))
assert len(rows) == len(questions), "row count must match test.csv"
print(f"wrote {len(rows)} rows")
print(rows[:5])